Private route-based policy iteration:125 own/public inputs, frozen retainedroute warmstart, confidence-weighted paired Monte Carlo teacher rewards, nine conservative neural alternatives vs retainedroute. Fresh game selection50m/final51m; smoke52m/53m. Free T4 only. No event submission.

In [ ]:
import pathlib,subprocess,json,os
work=pathlib.Path("/kaggle/temp/triad-uno-route-neural")
work.mkdir(parents=True,exist_ok=True)
sources={'uno_policy.py': '"""Linear action-ranking policy; inference uses filtered observations only."""\nfrom collections import Counter\nimport json\nfrom pathlib import Path\n\nFEATURE_NAMES = [\n    "play", "number", "skip", "reverse", "draw_two", "wild", "wild_draw_four",\n    "remaining_colour_count", "remaining_colour_fraction", "remaining_match_count",\n    "attack_next_threat", "reverse_escape_threat", "wild_when_small_hand",\n    "finish", "keep", "choose_colour", "next_can_finish", "own_hand_size",\n]\nINITIAL_WEIGHTS = [8., .5, -.2, -.2, -.2, -2., -3., 2., 1., 1., 4., 2., 2., 12., -4., 0., 0., 0.]\n\n\ndef action_features(observation, action):\n    hand = observation["hand"]\n    by_ref = {card["card_ref"]: card for card in hand}\n    card = action.get("card") or by_ref.get(action.get("card_ref")) or {}\n    kind = card.get("kind")\n    play = action["type"] == "play"\n    colour = action.get("color") or card.get("color") or observation.get("active_color")\n    remaining = [c for c in hand if not (play and c["card_ref"] == action.get("card_ref"))]\n    colours = Counter(c.get("color") for c in remaining if c.get("color"))\n    matching = sum(c.get("color") == colour or c.get("kind") in ("wild", "wild_draw_four")\n                   or (kind == "number" and c.get("kind") == "number" and c.get("value") == card.get("value"))\n                   or (kind not in (None, "number", "wild", "wild_draw_four") and c.get("kind") == kind)\n                   for c in remaining)\n    sizes = observation.get("hand_sizes", [len(hand)] * 4)\n    seat, direction = observation["seat"], observation.get("direction", 1)\n    active = [s for s in range(len(sizes)) if s not in observation.get("finished_order", [])]\n    def neighbour(d):\n        for distance in range(1, len(sizes)):\n            other = (seat + distance * d) % len(sizes)\n            if other in active:\n                return max(1, sizes[other])\n        return max(1, len(hand))\n    next_size, previous_size = neighbour(direction), neighbour(-direction)\n    threat = 1. / next_size\n    attack = play and kind in ("skip", "draw_two", "wild_draw_four")\n    return [float(play), float(kind == "number"), float(kind == "skip"),\n            float(kind == "reverse"), float(kind == "draw_two"), float(kind == "wild"),\n            float(kind == "wild_draw_four"), min(colours[colour], 12) / 4.,\n            colours[colour] / max(1, len(remaining)), min(matching, 12) / 4.,\n            float(attack) * threat,\n            float(play and kind == "reverse") * (threat - 1. / previous_size),\n            float(play and kind in ("wild", "wild_draw_four")) / max(1, len(hand)),\n            float(play and len(hand) == 1), float(action["type"] == "keep"),\n            float(action["type"] == "choose_color"), float(play) * float(next_size <= 2),\n            float(play) * min(len(hand), 24) / 12.]\n\n\nclass Policy:\n    def __init__(self, weights):\n        if len(weights) != len(FEATURE_NAMES):\n            raise ValueError("wrong policy feature count")\n        self.weights = [float(w) for w in weights]\n\n    def select(self, observation):\n        legal = observation["legal_actions"]\n        return max(legal, key=lambda action: sum(w * x for w, x in\n                   zip(self.weights, action_features(observation, action))))["action_id"]\n\n    def save(self, path, metadata):\n        Path(path).parent.mkdir(parents=True, exist_ok=True)\n        Path(path).write_text(json.dumps({"format": "triad-uno-linear-v1",\n            "features": FEATURE_NAMES, "weights": self.weights,\n            "training": metadata}, indent=2) + "\\n")\n', 'uno_policy_v2.py': '"""History-aware non-language UNO policy with public-information beliefs."""\nfrom collections import Counter, defaultdict\nimport json\nimport math\nfrom pathlib import Path\n\nfrom uno_policy import FEATURE_NAMES as V1_FEATURES, action_features as v1_features\n\nEXTRA_FEATURES = [\n    "successor_colour_absence", "successor_symbol_absence", "successor_colour_preference",\n    "successor_finish_risk", "remaining_colour_components", "largest_component_fraction",\n    "remaining_bridge_symbols", "remaining_duplicates", "remaining_colour_entropy",\n    "remaining_wilds", "chosen_colour_exhausted", "stack_penalty_play",\n    "successor_one_card_colour_absence", "successor_one_card_symbol_absence",\n]\nFEATURE_NAMES = V1_FEATURES + EXTRA_FEATURES\n\n\ndef symbol(card):\n    kind = card.get("kind")\n    if kind == "number":\n        return (kind, card.get("value"))\n    return (kind, None) if kind in ("skip", "reverse", "draw_two") else None\n\n\ndef public_beliefs(observation):\n    """Fold public events; a normal forced draw reveals pre-draw absences.\n\n    Unknown draws soften older evidence. Auto-kept or played drawn cards allow\n    exact post-draw absence deductions. Penalty draws do not imply any absence.\n    No simulator state, private opponent cards or seed is accessed here.\n    """\n    lack_colour, lack_symbol, preference, pending_draw = {}, {}, {}, {}\n    active_colour, top_symbol, pending_penalty = None, None, 0\n    for event in observation.get("history", []):\n        kind, seat = event.get("type"), event.get("seat")\n        if kind in ("game_start", "initial_discard"):\n            card = event.get("card", event.get("initial_discard", {})) or {}\n            active_colour = card.get("color", active_colour)\n            top_symbol = symbol(card)\n        elif kind == "initial_color_choice":\n            active_colour = event["color"]\n            preference[seat] = active_colour\n        elif kind in ("draw", "penalty_draw"):\n            count = event.get("drawn", 0) if kind == "draw" else event.get("count", 0)\n            for key in list(lack_colour):\n                if key[0] == seat:\n                    lack_colour[key] *= .75 ** count\n            for key in list(lack_symbol):\n                if key[0] == seat:\n                    lack_symbol[key] *= .92 ** count\n            if count:\n                preference.pop(seat, None)\n            if kind == "draw":\n                if active_colour:\n                    lack_colour[(seat, active_colour)] = .75 ** count\n                if top_symbol:\n                    lack_symbol[(seat, top_symbol)] = .92 ** count\n                pending_draw[seat] = (event.get("turn_number"), active_colour, top_symbol)\n            else:\n                pending_penalty = 0\n        elif kind in ("keep_auto", "play"):\n            card = event.get("card") or {}\n            if kind == "play":\n                colour, played_symbol = card.get("color"), symbol(card)\n                if colour:\n                    lack_colour[(seat, colour)] = 0.\n                if played_symbol:\n                    lack_symbol[(seat, played_symbol)] = 0.\n            drawn = pending_draw.get(seat)\n            if drawn and drawn[0] == event.get("turn_number") and (kind == "keep_auto" or event.get("from_draw")):\n                if drawn[1]:\n                    lack_colour[(seat, drawn[1])] = 1.\n                if drawn[2]:\n                    lack_symbol[(seat, drawn[2])] = 1.\n                pending_draw.pop(seat, None)\n            if kind == "play":\n                active_colour = event.get("active_color", card.get("color"))\n                top_symbol = symbol(card)\n                if card.get("kind") in ("wild", "wild_draw_four"):\n                    preference[seat] = active_colour\n                if "pending_penalty" in event:\n                    pending_penalty = event["pending_penalty"]\n    return {"colours": lack_colour, "symbols": lack_symbol, "preference": preference, "penalty": pending_penalty}\n\n\ndef structure(hand):\n    counts = Counter(c.get("color") for c in hand if c.get("color"))\n    groups = defaultdict(set)\n    for card in hand:\n        key = symbol(card)\n        if key and card.get("color"):\n            groups[key].add(card["color"])\n    adjacency = {colour: {colour} for colour in counts}\n    for colours in groups.values():\n        for colour in colours:\n            adjacency[colour].update(colours)\n    components, visited = [], set()\n    for colour in counts:\n        if colour in visited:\n            continue\n        frontier, connected = [colour], set()\n        while frontier:\n            current = frontier.pop()\n            if current not in connected:\n                connected.add(current)\n                frontier.extend(adjacency[current] - connected)\n        visited.update(connected)\n        components.append(sum(counts[c] for c in connected))\n    wilds = sum(c.get("kind") in ("wild", "wild_draw_four") for c in hand)\n    if wilds and components:\n        components = [sum(components)]\n    coloured = sum(counts.values())\n    entropy = -sum((n / coloured) * math.log(n / coloured) for n in counts.values()) if coloured else 0.\n    identical = Counter((c.get("color"), c.get("kind"), c.get("value")) for c in hand)\n    return [len(components) / 4., max(components, default=0) / max(1, len(hand)),\n            sum(len(colours) > 1 for colours in groups.values()) / 8.,\n            sum(n - 1 for n in identical.values()) / 4., entropy / 2., wilds / 4.]\n\n\ndef action_features(observation, action, beliefs=None):\n    beliefs = beliefs or public_beliefs(observation)\n    hand = observation["hand"]\n    play = action["type"] == "play"\n    card = action.get("card") or next((c for c in hand if c["card_ref"] == action.get("card_ref")), {})\n    kind = card.get("kind")\n    colour = action.get("color") or card.get("color") or observation.get("active_color")\n    remaining = [c for c in hand if not (play and c["card_ref"] == action.get("card_ref"))]\n    sizes, seat = observation["hand_sizes"], observation["seat"]\n    direction = observation.get("direction", 1)\n    active = [s for s in range(len(sizes)) if s not in observation.get("finished_order", [])]\n    if play and kind == "reverse":\n        direction = -direction\n    steps = 2 if play and kind in ("skip", "draw_two", "wild_draw_four") else 1\n    successor = seat\n    for _ in range(steps):\n        for distance in range(1, len(sizes) + 1):\n            other = (successor + distance * direction) % len(sizes)\n            if other in active:\n                successor = other\n                break\n    absence = beliefs["colours"].get((successor, colour), 0.)\n    absent_symbol = beliefs["symbols"].get((successor, symbol(card)), 0.)\n    threat = 1. / max(1, sizes[successor])\n    one_card = float(sizes[successor] == 1)\n    extras = [absence, absent_symbol, float(beliefs["preference"].get(successor) == colour),\n              threat * (1. - absence), *structure(remaining),\n              float(not any(c.get("color") == colour for c in remaining)),\n              float(play) * min(beliefs["penalty"], 24) / 12.,\n              one_card * absence, one_card * absent_symbol]\n    return v1_features(observation, action) + extras\n\n\nclass Policy:\n    def __init__(self, weights):\n        if len(weights) != len(FEATURE_NAMES) or not all(math.isfinite(float(w)) for w in weights):\n            raise ValueError("invalid v2 policy weights")\n        self.weights = list(weights)\n\n    def select(self, observation):\n        legal = observation["legal_actions"]\n        if len(legal) == 1:\n            return legal[0]["action_id"]\n        beliefs = public_beliefs(observation)\n        return max(legal, key=lambda action: sum(w * x for w, x in\n                   zip(self.weights, action_features(observation, action, beliefs))))["action_id"]\n\n    def save(self, path, metadata):\n        Path(path).parent.mkdir(parents=True, exist_ok=True)\n        Path(path).write_text(json.dumps({"format": "triad-uno-linear-v2", "features": FEATURE_NAMES,\n            "weights": self.weights, "training": metadata}, indent=2) + "\\n")\n', 'uno_route_policy.py': '"""Own-hand sequence features for a frozen learned UNO base policy.\n\nThe bounded search imagines subsequent own plays with no intervening top-card\nchanges. It is a hand-structure feature, not a claim about future opponents.\nOnly filtered observations are read. No simulator or hidden state is imported.\n"""\nfrom collections import Counter\nfrom functools import lru_cache\nimport math\nfrom uno_policy import Policy as Base, action_features as base_features, FEATURE_NAMES as BASE_NAMES\nfrom uno_policy_v2 import public_beliefs, symbol\n\nEXTRA_NAMES = ["sequence_fraction", "sequence_stranded", "ending_wild",\n               "next_colour_absence", "next_symbol_absence", "next_colour_preference"]\nFEATURE_NAMES = BASE_NAMES + EXTRA_NAMES\n\n\ndef route_length(hand, colour, top, max_states=128, max_depth=8):\n    """Longest discovered normal-play path, with a fixed deterministic budget."""\n    cards = tuple(sorted(Counter((c.get(\'color\') or \'\', c[\'kind\'], c.get(\'value\') if c.get(\'value\') is not None else -1)\n                                 for c in hand).items()))\n    types = tuple(c for c, _ in cards)\n    counts = tuple(n for _, n in cards)\n    visits = 0\n\n    @lru_cache(maxsize=256)\n    def walk(counts, active, last_kind, last_value, depth):\n        nonlocal visits\n        if not depth or visits >= max_states:\n            return 0\n        visits += 1\n        best = 0\n        active_count = sum(n for n, (c, _, _) in zip(counts, types) if c == active)\n        for i, ((c, kind, value), n) in enumerate(zip(types, counts)):\n            if not n:\n                continue\n            wild = kind in (\'wild\', \'wild_draw_four\')\n            legal = (kind == \'wild\' or (kind == \'wild_draw_four\' and not active_count)\n                     or c == active or (kind == \'number\' and last_kind == \'number\' and value == last_value)\n                     or kind in (\'skip\', \'reverse\', \'draw_two\') and kind == last_kind)\n            if not legal:\n                continue\n            reduced = counts[:i] + (n - 1,) + counts[i + 1:]\n            options = sorted({other[0] for other, number in zip(types, reduced) if number and other[0]}) if wild else [c]\n            if not options:\n                options = [active]\n            for chosen in options:\n                length = 1 + walk(reduced, chosen, kind, value, depth - 1)\n                best = max(best, length)\n                if best >= min(depth, sum(counts)):\n                    return best\n        return best\n    return walk(counts, colour or \'\', top.get(\'kind\'), top.get(\'value\', -1), min(max_depth, len(hand)))\n\n\ndef action_features(observation, action, beliefs=None, cache=None):\n    beliefs = beliefs or public_beliefs(observation)\n    play = action[\'type\'] == \'play\'\n    card = action.get(\'card\') or next((c for c in observation[\'hand\'] if c[\'card_ref\'] == action.get(\'card_ref\')), {})\n    colour = action.get(\'color\') or card.get(\'color\') or observation.get(\'active_color\')\n    remaining = [c for c in observation[\'hand\'] if not(play and c[\'card_ref\'] == action.get(\'card_ref\'))]\n    top = card if play else observation.get(\'top_discard\', {})\n    key = (tuple(sorted((c.get(\'color\') or \'\', c[\'kind\'], c.get(\'value\', -1) if c.get(\'value\') is not None else -1)\n                        for c in remaining)), colour, top.get(\'kind\'), top.get(\'value\'))\n    if cache is not None and key in cache:\n        length = cache[key]\n    else:\n        length = route_length(remaining, colour, top)\n        if cache is not None:\n            cache[key] = length\n    sizes = observation[\'hand_sizes\']\n    direction = observation.get(\'direction\', 1)\n    if play and card.get(\'kind\') == \'reverse\':\n        direction = -direction\n    active = [s for s in range(len(sizes)) if s not in observation.get(\'finished_order\', [])]\n    successor = observation[\'seat\']\n    steps = 2 if play and card.get(\'kind\') in (\'skip\', \'draw_two\', \'wild_draw_four\') else 1\n    for _ in range(steps):\n        successor = next((successor + distance * direction) % len(sizes) for distance in range(1, len(sizes) + 1)\n                         if (successor + distance * direction) % len(sizes) in active)\n    denom = min(8, len(remaining))\n    extras = [length / max(1, denom), (denom - length) / 8.,\n              float(len(remaining) == 1 and remaining[0][\'kind\'] in (\'wild\', \'wild_draw_four\')),\n              beliefs[\'colours\'].get((successor, colour), 0.),\n              beliefs[\'symbols\'].get((successor, symbol(top)), 0.),\n              float(beliefs[\'preference\'].get(successor) == colour)]\n    return base_features(observation, action) + extras\n\n\nclass Policy:\n    def __init__(self, weights):\n        if len(weights) != len(FEATURE_NAMES) or not all(math.isfinite(w) for w in weights):\n            raise ValueError(\'invalid route policy weights\')\n        self.weights = list(weights)\n\n    def select(self, observation):\n        legal = observation[\'legal_actions\']\n        if len(legal) == 1:\n            return legal[0][\'action_id\']\n        beliefs, cache = public_beliefs(observation), {}\n        return max(legal, key=lambda a: sum(w * x for w, x in zip(self.weights, action_features(observation, a, beliefs, cache))))[\'action_id\']\n', 'uno_policy_features_v3.py': '"""Richer numeric own-hand encoding, canonicalized under colour permutations."""\nfrom collections import Counter\nimport math\nfrom uno_policy_v2 import FEATURE_NAMES as V2_FEATURES, action_features as v2_features, public_beliefs, symbol\n\nKINDS=[(\'number\',n) for n in range(10)]+[(\'skip\',None),(\'reverse\',None),(\'draw_two\',None)]\nFEATURE_NAMES=V2_FEATURES+[\n f\'remaining_colour_slot_{slot}_{kind}_{value}\' for slot in range(4) for kind,value in KINDS]+[\n \'remaining_wild_count\',\'remaining_wild_draw_four_count\']+[\n f\'action_number_{number}\' for number in range(10)]+[\n f\'opponent_slot_{slot}_{name}\' for slot in range(3) for name in (\'present\',\'hand_size\',\'one_card\',\'colour_absence\',\'symbol_absence\',\'colour_preference\')]+[\n \'selected_is_active_colour\',\'pending_penalty_size\',\'turn_progress\',\'active_opponents\',\'closest_opponent_risk\']\n\n\ndef action_features(observation,action,beliefs=None):\n beliefs=beliefs or public_beliefs(observation)\n hand=observation[\'hand\'];play=action[\'type\']==\'play\'\n card=action.get(\'card\') or next((c for c in hand if c[\'card_ref\']==action.get(\'card_ref\')), {})\n selected=action.get(\'color\') or card.get(\'color\') or observation.get(\'active_color\')\n remaining=[c for c in hand if not(play and c[\'card_ref\']==action.get(\'card_ref\'))]\n counts=Counter((c.get(\'color\'),c.get(\'kind\'),c.get(\'value\') if c.get(\'kind\')==\'number\' else None) for c in remaining)\n colours=[\'red\',\'yellow\',\'green\',\'blue\']\n order=[]\n for colour in (selected,observation.get(\'active_color\')):\n  if colour in colours and colour not in order:order.append(colour)\n def key(colour):\n  vector=tuple(counts[(colour,kind,value)] for kind,value in KINDS)\n  return sum(vector),vector\n order+=sorted((c for c in colours if c not in order),key=key,reverse=True)\n vector=[counts[(colour,kind,value)]/2. for colour in order for kind,value in KINDS]\n vector.extend(sum(c.get(\'kind\')==kind for c in remaining)/4. for kind in (\'wild\',\'wild_draw_four\'))\n vector.extend(float(card.get(\'kind\')==\'number\' and card.get(\'value\')==n) for n in range(10))\n seat=observation[\'seat\'];sizes=observation[\'hand_sizes\'];direction=observation.get(\'direction\',1)\n if play and card.get(\'kind\')==\'reverse\':direction=-direction\n finished=set(observation.get(\'finished_order\',[]))\n opponents=[(seat+i*direction)%len(sizes) for i in range(1,len(sizes)) if (seat+i*direction)%len(sizes) not in finished]\n for slot in range(3):\n  if slot>=len(opponents):vector.extend([0.]*6);continue\n  other=opponents[slot]\n  vector.extend([1.,sizes[other]/20.,float(sizes[other]==1),\n   beliefs[\'colours\'].get((other,selected),0.),beliefs[\'symbols\'].get((other,symbol(card)),0.),\n   float(beliefs[\'preference\'].get(other)==selected)])\n vector.extend([float(selected==observation.get(\'active_color\')),min(48,beliefs[\'penalty\'])/12.,\n  min(1000,observation.get(\'turn_number\',0))/1000.,len(opponents)/3.,\n  1./max(1,sizes[opponents[0]]) if opponents else 0.])\n return v2_features(observation,action,beliefs)+vector\n', 'uno_policy_features_v4.py': '"""Full own/public encoding plus exact retained hand-route feature values."""\nfrom uno_policy_features_v3 import FEATURE_NAMES as RICH_NAMES, action_features as rich_features\nfrom uno_route_policy import EXTRA_NAMES, action_features as route_features\n\nFEATURE_NAMES=RICH_NAMES+[\'route_\'+name for name in EXTRA_NAMES]\n\n\ndef action_features(observation,action,beliefs=None):\n    return rich_features(observation,action,beliefs)+route_features(observation,action,beliefs)[-len(EXTRA_NAMES):]\n', 'uno_mlp_policy.py': '"""Small non-language action scorer with a learned residual over the incumbent."""\nimport json\nimport math\nfrom pathlib import Path\nfrom uno_policy_v2 import FEATURE_NAMES, action_features, public_beliefs\n\n\nclass Policy:\n    def __init__(self, model):\n        self.feature_names = FEATURE_NAMES\n        self.action_features = action_features\n        if model.get(\'format\') == \'triad-uno-residual-mlp-v3\':\n            from uno_policy_features_v4 import FEATURE_NAMES as rich_names, action_features as rich_features\n            self.feature_names = rich_names\n            self.action_features = rich_features\n        elif model.get(\'format\') == \'triad-uno-residual-mlp-v2\':\n            from uno_policy_features_v3 import FEATURE_NAMES as rich_names, action_features as rich_features\n            self.feature_names = rich_names\n            self.action_features = rich_features\n        elif model.get(\'format\') != \'triad-uno-residual-mlp-v1\':\n            raise ValueError(\'incompatible model format\')\n        if model.get(\'features\') != self.feature_names:\n            raise ValueError(\'incompatible model format/features\')\n        self.model = model\n        self.base = model[\'base_weights\']\n        self.w1, self.b1 = model[\'hidden_weight\'], model[\'hidden_bias\']\n        self.w2, self.b2 = model[\'output_weight\'], model[\'output_bias\']\n        width = len(self.b1)\n        if len(self.base) != len(self.feature_names) or len(self.w1) != width or len(self.w2) != width or not 1 <= width <= 128:\n            raise ValueError(\'invalid model dimensions\')\n        if any(len(row) != len(self.feature_names) for row in self.w1):\n            raise ValueError(\'invalid hidden dimensions\')\n        values = self.base + self.b1 + self.w2 + [self.b2] + [x for row in self.w1 for x in row]\n        if not all(math.isfinite(float(x)) for x in values):\n            raise ValueError(\'nonfinite model parameter\')\n\n    def score_features(self, vector):\n        hidden = [max(0., sum(w*x for w,x in zip(row,vector)) + bias) for row,bias in zip(self.w1,self.b1)]\n        return sum(w*x for w,x in zip(self.base,vector)) + sum(w*x for w,x in zip(self.w2,hidden)) + self.b2\n\n    def select(self, observation):\n        legal = observation[\'legal_actions\']\n        if len(legal) == 1:\n            return legal[0][\'action_id\']\n        beliefs = public_beliefs(observation)\n        return max(legal,key=lambda action:self.score_features(self.action_features(observation,action,beliefs)))[\'action_id\']\n\n    def save(self, path):\n        Path(path).write_text(json.dumps(self.model,indent=2)+\'\\n\')\n\n\nclass FastPolicy(Policy):\n    """NumPy batch scoring for training evaluation only; never shipped."""\n    def __init__(self, model):\n        super().__init__(model)\n        import numpy as np\n        self.np = np\n        self.fast_base = np.asarray(self.base, dtype=float)\n        self.fast_w1 = np.asarray(self.w1, dtype=float)\n        self.fast_b1 = np.asarray(self.b1, dtype=float)\n        self.fast_w2 = np.asarray(self.w2, dtype=float)\n\n    def select(self, observation):\n        legal = observation[\'legal_actions\']\n        if len(legal) == 1:\n            return legal[0][\'action_id\']\n        beliefs = public_beliefs(observation)\n        x = self.np.asarray([self.action_features(observation,a,beliefs) for a in legal], dtype=float)\n        hidden = self.np.maximum(0., x @ self.fast_w1.T + self.fast_b1)\n        scores = x @ self.fast_base + hidden @ self.fast_w2 + self.b2\n        return legal[int(self.np.argmax(scores))][\'action_id\']\n', 'train_uno_league.py': '"""PSRO-inspired archive league with paired seat blocks and separate test seeds.\n\nThis is a practical population training method, not an exact equilibrium solver.\nAll policies act on filtered observations; simulator state only computes rewards.\n"""\nimport argparse\nimport concurrent.futures\nimport json\nimport multiprocessing\nimport os\nfrom pathlib import Path\nimport random\nimport sys\nimport time\n\nimport numpy as np\n\nROOT = Path(__file__).resolve().parent\nSDK = Path(os.environ.get("UNO_SDK", ROOT.parent / "unoSDK"))\nsys.path.insert(0, str(SDK / "local_evaluator" / "src"))\nfrom triad_uno import engine\nfrom triad_uno.baselines import HeuristicAgent, RandomAgent\nfrom uno_policy import Policy as V1, INITIAL_WEIGHTS\nfrom uno_policy_v2 import Policy as V2, FEATURE_NAMES\n\n\nclass Starter:\n    def select(self, observation):\n        legal = observation["legal_actions"]\n        return next((a["action_id"] for a in legal if a["type"] == "play"), legal[0]["action_id"])\n\n\ndef make_policy(spec, seed):\n    if spec["kind"] == "route":\n        from uno_route_policy import Policy as RoutePolicy\n        return RoutePolicy(spec["weights"])\n    if spec["kind"] == "mlp":\n        from uno_mlp_policy import FastPolicy\n        return FastPolicy(spec["model"])\n    if spec["kind"] == "v1":\n        return V1(spec["weights"])\n    if spec["kind"] == "v2":\n        return V2(spec["weights"])\n    if spec["kind"] == "random":\n        return RandomAgent(seed)\n    if spec["kind"] == "starter":\n        return Starter()\n    return HeuristicAgent(seed)\n\n\ndef score_game(seed, seat, candidate, rivals):\n    actors = [None] * 4\n    actors[seat] = make_policy(candidate, seed * 10 + 9)\n    for offset, spec in enumerate(rivals, 1):\n        actors[(seat + offset) % 4] = make_policy(spec, seed * 10 + offset)\n    state = engine.new_game(engine.GameConfig(max_turns=1000), seed)\n    while state.phase != engine.FINISHED and seat not in state.finished_set:\n        current = state.current_seat\n        observation = engine.observe(state, current)\n        selected = actors[current].select(observation)\n        action = engine.engine_action_from_id(state, current, selected)\n        if action is None:\n            raise RuntimeError("policy selected an illegal action")\n        if not engine.step(state, action).ok:\n            raise RuntimeError("policy transition failed")\n    if seat in state.finished_order:\n        placement = state.finished_order.index(seat)\n        return state.config.position_points[placement], float(placement == 0)\n    outcome = state.outcome\n    remaining = state.config.position_points[len(outcome.placements):]\n    return sum(remaining) / len(outcome.unfinished), 0.\n\n\ndef evaluate_job(job):\n    candidate, seeds, pool, mode = job\n    blocks, win_blocks = [], []\n    for seed in seeds:\n        rng = random.Random(seed ^ 0x4C4541475545)\n        if mode == "league":\n            rivals = [rng.choice(pool) for _ in range(3)]\n        else:\n            rivals = [{"kind": mode}] * 3\n        outcomes = [score_game(seed, seat, candidate, rivals) for seat in range(4)]\n        blocks.append(sum(row[0] for row in outcomes) / 4.)\n        win_blocks.append(sum(row[1] for row in outcomes) / 4.)\n    return {"points": float(np.mean(blocks)), "win_rate": float(np.mean(win_blocks)),\n            "games": 4 * len(seeds), "block_points": blocks, "block_wins": win_blocks}\n\n\ndef paired_interval(candidate, incumbent, seed=20261014):\n    differences = np.asarray(candidate["block_points"]) - np.asarray(incumbent["block_points"])\n    rng = np.random.default_rng(seed)\n    if len(differences) < 2:\n        return {"mean_gain": float(np.mean(differences)), "ci95": None}\n    means = differences[rng.integers(0, len(differences), (5000, len(differences)))].mean(axis=1)\n    return {"mean_gain": float(differences.mean()),\n            "ci95": [float(v) for v in np.quantile(means, [.025, .975])],\n            "unit": "points per game", "sampling_unit": "four-seat seed block"}\n\n\ndef summarize(result):\n    return {k: v for k, v in result.items() if not k.startswith("block_")}\n\n\ndef main():\n    parser = argparse.ArgumentParser()\n    parser.add_argument("--incumbent", type=Path, default=SDK / "models" / "policy.json")\n    parser.add_argument("--iterations", type=int, default=40)\n    parser.add_argument("--population", type=int, default=48)\n    parser.add_argument("--train-blocks", type=int, default=16)\n    parser.add_argument("--validation-blocks", type=int, default=128)\n    parser.add_argument("--test-blocks", type=int, default=256)\n    parser.add_argument("--workers", type=int, default=min(4, os.cpu_count() or 1))\n    parser.add_argument("--seed", type=int, default=20261015)\n    parser.add_argument("--output", type=Path, default=ROOT / "uno-league-output")\n    args = parser.parse_args()\n    if min(args.iterations, args.population, args.train_blocks, args.validation_blocks, args.test_blocks, args.workers) < 1:\n        parser.error("counts must be positive")\n    args.output.mkdir(parents=True, exist_ok=True)\n    document = json.loads(args.incumbent.read_text())\n    incumbent = {"kind": "v1", "weights": document["weights"]}\n    mean = np.asarray(document["weights"] + [0.] * (len(FEATURE_NAMES) - len(document["weights"])))\n    best = mean.copy()\n    sigma = np.full(len(mean), 1.5)\n    rng = np.random.default_rng(args.seed)\n    attacking = list(INITIAL_WEIGHTS)\n    for i in (2, 4, 6, 10):\n        attacking[i] += 2.\n    preserve = list(INITIAL_WEIGHTS)\n    preserve[5] -= 2.\n    preserve[6] -= 2.\n    reference = [{"kind": "starter"}, {"kind": "heuristic"}, {"kind": "random"},\n                 incumbent, {"kind": "v1", "weights": INITIAL_WEIGHTS},\n                 {"kind": "v1", "weights": attacking}, {"kind": "v1", "weights": preserve}]\n    archive = list(reference)\n    selection_seeds = list(range(6_000_000, 6_000_000 + args.validation_blocks))\n    history, started = [], time.time()\n    context = multiprocessing.get_context("spawn")\n    with concurrent.futures.ProcessPoolExecutor(max_workers=args.workers, mp_context=context) as pool:\n        baseline = list(pool.map(evaluate_job, [(incumbent, selection_seeds, reference, "league")]))[0]\n        selected_score = baseline["points"]\n        for generation in range(args.iterations):\n            if generation and generation % 10 == 0:\n                mean, sigma = best.copy(), np.full(len(mean), 1.5)\n            population = rng.normal(mean, sigma, (args.population, len(mean)))\n            # Broad probes keep the population from collapsing into one local basin.\n            probes = max(1, args.population // 5)\n            population[-probes:] = rng.normal(best, 3., (probes, len(mean)))\n            population = np.clip(population, -20., 20.)\n            population[0] = mean\n            if args.population > 1:\n                population[1] = best\n            seeds = list(range(30_000 + generation * args.train_blocks, 30_000 + (generation + 1) * args.train_blocks))\n            scores = list(pool.map(evaluate_job,\n                [({"kind": "v2", "weights": w.tolist()}, seeds, archive, "league") for w in population]))\n            ranked = sorted(range(len(scores)), key=lambda i: (scores[i]["points"], scores[i]["win_rate"]), reverse=True)\n            elite = population[ranked[:max(2, args.population // 5)]]\n            mean = .35 * mean + .65 * elite.mean(axis=0)\n            sigma = np.maximum(.3, .35 * sigma + .65 * elite.std(axis=0))\n            candidate = population[ranked[0]]\n            spec = {"kind": "v2", "weights": candidate.tolist()}\n            validation = list(pool.map(evaluate_job, [(spec, selection_seeds, reference, "league")]))[0]\n            if validation["points"] > selected_score:\n                best, selected_score = candidate.copy(), validation["points"]\n            if generation % 4 == 3 and not any(s.get("kind") == "v2" and np.linalg.norm(np.asarray(s["weights"]) - candidate) < .2 for s in archive):\n                archive.append(spec)\n                archive = reference + archive[len(reference):][-12:]\n            record = {"generation": generation + 1, "train": summarize(scores[ranked[0]]),\n                      "validation": summarize(validation), "paired_validation_gain": paired_interval(validation, baseline),\n                      "selected_points": selected_score, "archive_size": len(archive),\n                      "elapsed_seconds": round(time.time() - started, 1)}\n            history.append(record)\n            print(json.dumps(record), flush=True)\n            V2(best).save(args.output / "uno-policy-v2.json", {"method": "archive-league cross-entropy policy optimization",\n                "seed": args.seed, "generations": generation + 1, "population": args.population,\n                "train_blocks": args.train_blocks, "validation_blocks": args.validation_blocks,\n                "selected_validation_points": selected_score, "incumbent_validation_points": baseline["points"],\n                "sdk_commit": "0cda3acc1bb5c05fa1803a8fbd7761770dc6d5fe"})\n            (args.output / "training-progress.json").write_text(json.dumps(history, indent=2) + "\\n")\n            (args.output / "archive.json").write_text(json.dumps(archive, indent=2) + "\\n")\n        test_seeds = list(range(7_000_000, 7_000_000 + args.test_blocks))\n        chosen = {"kind": "v2", "weights": best.tolist()}\n        modes = ("league", "starter", "heuristic", "random")\n        jobs = [(spec, test_seeds, reference, mode) for mode in modes for spec in (chosen, incumbent)]\n        results = list(pool.map(evaluate_job, jobs))\n        report = {mode: {"candidate": summarize(results[2*i]), "incumbent": summarize(results[2*i+1]),\n                        "paired_gain": paired_interval(results[2*i], results[2*i+1])}\n                  for i, mode in enumerate(modes)}\n        (args.output / "evaluation.json").write_text(json.dumps(report, indent=2) + "\\n")\n        print(json.dumps({"held_out_evaluation": report}), flush=True)\n\n\nif __name__ == "__main__":\n    main()\n', 'train_uno_residual.py': '"""GPU pairwise policy distillation, followed by real held-out SDK game tests.\n\nSplit complete source games, not actions, for supervised validation. Preserve\nincumbent until a separately tested deterministic candidate shows useful gains.\n"""\nimport argparse\nimport concurrent.futures\nimport copy\nimport json\nimport multiprocessing\nimport os\nfrom pathlib import Path\nimport time\nimport numpy as np\nimport torch\nfrom torch import nn\nfrom torch.nn import functional as F\nfrom uno_policy import INITIAL_WEIGHTS\nfrom uno_policy_v2 import FEATURE_NAMES\nfrom uno_mlp_policy import Policy\nfrom train_uno_league import evaluate_job, paired_interval, summarize\n\n\nclass Residual(nn.Module):\n    def __init__(self, base, width):\n        super().__init__()\n        self.register_buffer(\'base\',torch.tensor(base,dtype=torch.float32))\n        self.hidden=nn.Linear(len(base),width)\n        self.output=nn.Linear(width,1)\n        nn.init.zeros_(self.output.weight);nn.init.zeros_(self.output.bias)\n\n    def forward(self,x):\n        residual=self.output(F.relu(self.hidden(x))).squeeze(-1)\n        return x@self.base+residual,residual\n\n\ndef export(model,base,metadata):\n    names=metadata[\'source_dataset\'][\'input_features\']\n    version={3:\'v2\',4:\'v3\'}.get(metadata[\'source_dataset\'].get(\'feature_version\'),\'v1\')\n    return {\'format\':\'triad-uno-residual-mlp-\'+version,\'features\':names,\'base_weights\':base,\n            \'hidden_weight\':model.hidden.weight.detach().cpu().tolist(),\n            \'hidden_bias\':model.hidden.bias.detach().cpu().tolist(),\n            \'output_weight\':model.output.weight.detach().cpu().flatten().tolist(),\n            \'output_bias\':float(model.output.bias.detach().cpu()[0]),\'training\':metadata}\n\n\ndef make_pairs(data,seed):\n    # NPZ access decompresses a member. Load each array once, rather than once\n    # per state/pair, especially the new Monte Carlo trial matrix.\n    game_ids=data[\'game_ids\'];targets=data[\'targets\'];offsets=data[\'offsets\']\n    trial_targets=data[\'trial_targets\'] if \'trial_targets\' in data else None\n    ids=np.unique(game_ids)\n    rng=np.random.default_rng(seed);rng.shuffle(ids)\n    val_ids=set(ids[:max(1,len(ids)//5)].tolist())\n    train,val=[],[]\n    for state,(a,b) in enumerate(zip(offsets[:-1],offsets[1:])):\n        target=val if int(game_ids[state]) in val_ids else train\n        q=targets[a:b]\n        for i in range(len(q)):\n            for j in range(i+1,len(q)):\n                gap=float(q[i]-q[j])\n                if abs(gap)>=.5:\n                    weight=min(1.,abs(gap)/3.)\n                    if trial_targets is not None and trial_targets.shape[1] > 1:\n                        differences=trial_targets[a+i]-trial_targets[a+j]\n                        stderr=float(np.std(differences,ddof=1)/np.sqrt(len(differences)))\n                        # Reward differences sharing future draws, not independent\n                        # action variances, provide the relevant noise estimate.\n                        weight*=max(.05,1.-stderr/abs(gap))\n                    target.append((a+i,a+j,1./(1.+np.exp(-gap/.75)),weight))\n    if not train or not val:raise ValueError(\'not enough informative game-held-out pairs\')\n    return np.asarray(train,dtype=np.float64),np.asarray(val,dtype=np.float64),len(ids)-len(val_ids),len(val_ids)\n\n\ndef loss(model,x,pairs,regularization):\n    first=pairs[:,0].long();second=pairs[:,1].long()\n    scores_a,residual_a=model(x[first]);scores_b,residual_b=model(x[second])\n    point=F.binary_cross_entropy_with_logits(scores_a-scores_b,pairs[:,2],reduction=\'none\')\n    rank=(point*pairs[:,3]).sum()/pairs[:,3].sum()\n    regularizer=regularization*(residual_a.square().mean()+residual_b.square().mean())/2.\n    return rank+regularizer,rank\n\n\ndef main():\n    parser=argparse.ArgumentParser()\n    parser.add_argument(\'--data\',type=Path,required=True)\n    parser.add_argument(\'--evaluation-incumbent\',type=Path,help=\'optional current retained policy for actual-game comparisons\')\n    parser.add_argument(\'--epochs\',type=int,default=100)\n    parser.add_argument(\'--width\',type=int,default=32)\n    parser.add_argument(\'--batch\',type=int,default=2048)\n    parser.add_argument(\'--workers\',type=int,default=4)\n    parser.add_argument(\'--validation-blocks\',type=int,default=128)\n    parser.add_argument(\'--test-blocks\',type=int,default=256)\n    parser.add_argument(\'--validation-seed-start\',type=int,default=11_000_000)\n    parser.add_argument(\'--test-seed-start\',type=int,default=12_000_000)\n    parser.add_argument(\'--seed\',type=int,default=20261017)\n    parser.add_argument(\'--device\',default=\'cuda\')\n    parser.add_argument(\'--residual-scales\',default=\'1\',help=\'comma-separated conservative learned-residual scales\')\n    parser.add_argument(\'--output\',type=Path,default=Path(__file__).parent/\'uno-residual-output\')\n    args=parser.parse_args()\n    scales=[float(s) for s in args.residual_scales.split(\',\')]\n    if not scales or any(not np.isfinite(s) or not 0<s<=1 for s in scales):parser.error(\'residual scales must be finite in(0,1]\')\n    if min(args.epochs,args.width,args.batch,args.workers,args.validation_blocks,args.test_blocks)<1:parser.error(\'positive counts required\')\n    if args.device==\'cuda\' and not torch.cuda.is_available():raise RuntimeError(\'requested authorized GPU unavailable; do not silently select other hardware\')\n    torch.set_num_threads(1)\n    args.output.mkdir(parents=True,exist_ok=True)\n    metadata=json.loads((args.data.parent/\'metadata.json\').read_text())\n    names=FEATURE_NAMES\n    if metadata.get(\'feature_version\')==4:\n        from uno_policy_features_v4 import FEATURE_NAMES as rich_names\n        names=rich_names\n    elif metadata.get(\'feature_version\')==3:\n        from uno_policy_features_v3 import FEATURE_NAMES as rich_names\n        names=rich_names\n    if metadata[\'input_features\']!=names:raise ValueError(\'feature schema mismatch\')\n    data=np.load(args.data)\n    if metadata.get(\'feature_version\')==4:\n        weights=metadata[\'incumbent\'][\'weights\']\n        if metadata[\'incumbent\'][\'format\']!=\'triad-uno-route-linear-v1\' or len(weights)!=24:raise ValueError(\'route feature schema requires retained route teacher\')\n        base=weights[:18]+[0.]*(len(names)-24)+weights[18:]\n    else:\n        base=metadata[\'incumbent\'][\'weights\']+[0.]*(len(names)-len(metadata[\'incumbent\'][\'weights\']))\n    train,val,train_games,val_games=make_pairs(data,args.seed)\n    x=torch.as_tensor(data[\'features\'],dtype=torch.float32,device=args.device)\n    train=torch.as_tensor(train,dtype=torch.float32,device=args.device)\n    val=torch.as_tensor(val,dtype=torch.float32,device=args.device)\n    print(json.dumps({\'device\':args.device,\'devices\':torch.cuda.device_count(),\'states\':len(data[\'game_ids\']),\n                      \'actions\':len(x),\'train_games\':train_games,\'validation_games\':val_games,\n                      \'train_pairs\':len(train),\'validation_pairs\':len(val)}),flush=True)\n    candidates=[];history=[];started=time.time()\n    for index,regularization in enumerate((.01,.03,.1)):\n        torch.manual_seed(args.seed+index)\n        model=Residual(base,args.width).to(args.device)\n        optimizer=torch.optim.AdamW(model.parameters(),lr=.003,weight_decay=.0001)\n        with torch.no_grad():\n            _,baseline_rank=loss(model,x,val,regularization)\n        best_loss=float(baseline_rank);best_epoch=0;best=copy.deepcopy(model.state_dict())\n        for epoch in range(args.epochs):\n            model.train();order=torch.randperm(len(train),device=args.device)\n            for start in range(0,len(order),args.batch):\n                batch=train[order[start:start+args.batch]]\n                objective,_=loss(model,x,batch,regularization)\n                optimizer.zero_grad(set_to_none=True);objective.backward()\n                nn.utils.clip_grad_norm_(model.parameters(),5.)\n                optimizer.step()\n            model.eval()\n            with torch.no_grad():\n                objective,rank=loss(model,x,val,regularization)\n            current=float(rank)\n            if current<best_loss:\n                best_loss,best_epoch,best=current,epoch+1,copy.deepcopy(model.state_dict())\n            if (epoch+1)%10==0:\n                row={\'regularization\':regularization,\'epoch\':epoch+1,\'validation_rank_loss\':current,\'best_rank_loss\':best_loss,\'seconds\':round(time.time()-started,1)}\n                history.append(row);print(json.dumps(row),flush=True)\n        model.load_state_dict(best)\n        document=export(model,base,{\'method\':\'paired counterfactual policy distillation with residual scorer\',\n            \'seed\':args.seed+index,\'regularization\':regularization,\'best_epoch\':best_epoch,\n            \'validation_rank_loss\':best_loss,\'baseline_validation_rank_loss\':float(baseline_rank),\n            \'training_games\':train_games,\'validation_games\':val_games,\'source_dataset\':metadata,\n            \'sdk_commit\':metadata[\'sdk_commit\']})\n        check=data[\'features\'][:min(200,len(data[\'features\']))]\n        with torch.no_grad():\n            tensor=torch.as_tensor(check,dtype=torch.float32,device=args.device)\n            _,residual=model(tensor)\n            base_scores=tensor@model.base\n        for scale in scales:\n            scaled=copy.deepcopy(document)\n            scaled[\'output_weight\']=[scale*w for w in document[\'output_weight\']]\n            scaled[\'output_bias\']=scale*document[\'output_bias\']\n            scaled[\'training\'][\'residual_scale\']=scale\n            deployed=Policy(scaled) # Deployment dimensions/finite parameters.\n            expected=(base_scores+scale*residual).cpu().numpy()\n            actual=np.asarray([deployed.score_features(row.tolist()) for row in check])\n            max_error=float(np.max(np.abs(actual-expected)))\n            if max_error>1e-4:raise RuntimeError(\'exported standard-library scorer differs from training network\')\n            candidate_index=len(candidates)\n            print(json.dumps({\'candidate\':candidate_index,\'residual_scale\':scale,\'export_max_score_error\':max_error}),flush=True)\n            path=args.output/f\'candidate-{candidate_index}.json\';path.write_text(json.dumps(scaled,indent=2)+\'\\n\')\n            candidates.append(scaled)\n        del model,optimizer\n    (args.output/\'training-progress.json\').write_text(json.dumps(history,indent=2)+\'\\n\')\n    incumbent_document=metadata[\'incumbent\']\n    if args.evaluation_incumbent:\n        incumbent_document=json.loads(args.evaluation_incumbent.read_text())\n    kind={\'triad-uno-linear-v1\':\'v1\',\'triad-uno-route-linear-v1\':\'route\'}.get(incumbent_document.get(\'format\'))\n    if kind is None:raise ValueError(\'unsupported actual-game evaluation incumbent\')\n    incumbent={\'kind\':kind,\'weights\':incumbent_document[\'weights\']}\n    source_kind={\'triad-uno-linear-v1\':\'v1\',\'triad-uno-route-linear-v1\':\'route\'}.get(metadata[\'incumbent\'][\'format\'])\n    if source_kind is None:raise ValueError(\'unsupported rollout teacher format\')\n    source_incumbent={\'kind\':source_kind,\'weights\':metadata[\'incumbent\'][\'weights\']}\n    attacking=list(INITIAL_WEIGHTS)\n    for i in (2,4,6,10):attacking[i]+=2.\n    preserve=list(INITIAL_WEIGHTS);preserve[5]-=2.;preserve[6]-=2.\n    reference=[{\'kind\':\'starter\'},{\'kind\':\'heuristic\'},{\'kind\':\'random\'},incumbent,\n               {\'kind\':\'v1\',\'weights\':INITIAL_WEIGHTS},{\'kind\':\'v1\',\'weights\':attacking},\n               {\'kind\':\'v1\',\'weights\':preserve}]\n    if source_incumbent != incumbent:reference.insert(4,source_incumbent)\n    specs=[incumbent]+[{\'kind\':\'mlp\',\'model\':m} for m in candidates]\n    seeds=list(range(args.validation_seed_start,args.validation_seed_start+args.validation_blocks))\n    with concurrent.futures.ProcessPoolExecutor(max_workers=args.workers,mp_context=multiprocessing.get_context(\'spawn\')) as pool:\n        results=list(pool.map(evaluate_job,[(s,seeds,reference,\'league\') for s in specs]))\n        chosen=max(range(len(results)),key=lambda i:results[i][\'points\'])\n        selection={\'selected\':chosen,\'scores\':[summarize(r) for r in results],\n                   \'paired_gains\':[paired_interval(r,results[0]) for r in results[1:]],\'evaluation_incumbent\':incumbent_document}\n        (args.output/\'selection.json\').write_text(json.dumps(selection,indent=2)+\'\\n\');print(json.dumps({\'game_selection\':selection}),flush=True)\n        selected=specs[chosen]\n        if chosen:\n            (args.output/\'uno-policy-mlp.json\').write_text(json.dumps(candidates[chosen-1],indent=2)+\'\\n\')\n        else:\n            (args.output/\'retained-incumbent.json\').write_text(json.dumps(incumbent_document,indent=2)+\'\\n\')\n        seeds=list(range(args.test_seed_start,args.test_seed_start+args.test_blocks));modes=(\'league\',\'starter\',\'heuristic\',\'random\')\n        results=list(pool.map(evaluate_job,[(s,seeds,reference,m) for m in modes for s in (selected,incumbent)]))\n        report={mode:{\'candidate\':summarize(results[2*i]),\'incumbent\':summarize(results[2*i+1]),\n                      \'paired_gain\':paired_interval(results[2*i],results[2*i+1])} for i,mode in enumerate(modes)}\n        (args.output/\'evaluation.json\').write_text(json.dumps(report,indent=2)+\'\\n\');print(json.dumps({\'held_out_evaluation\':report}),flush=True)\n\n\nif __name__==\'__main__\':main()\n'}
for name,source in sources.items():(work/name).write_text(source)

current_incumbent={'format': 'triad-uno-route-linear-v1', 'features': ['play', 'number', 'skip', 'reverse', 'draw_two', 'wild', 'wild_draw_four', 'remaining_colour_count', 'remaining_colour_fraction', 'remaining_match_count', 'attack_next_threat', 'reverse_escape_threat', 'wild_when_small_hand', 'finish', 'keep', 'choose_colour', 'next_can_finish', 'own_hand_size', 'sequence_fraction', 'sequence_stranded', 'ending_wild', 'next_colour_absence', 'next_symbol_absence', 'next_colour_preference'], 'weights': [8.022466349142684, 0.44153069585311683, -0.06834035646341297, 1.4643534535685507, -2.338619866848944, -3.4540620544700498, -4.276279368110987, 1.6602353386949849, 1.0165100023007814, 0.901341302040876, 3.853185081203389, 2.4436903583872347, 1.7223969671761579, 11.872419584625783, -4.598739912392293, 3.347870880522802, 0.32778952616834095, -0.24548152213113314, 0.054801310646275025, -1.1349214815798592, -0.04225874297028254, 1.1191018681573717, 0.21891410472226475, -0.08122782541248305], 'training': {'method': 'frozen-base residual CEM, own-hand bounded sequence features', 'seed': 20261019, 'iterations': 24, 'sdk_commit': '0cda3acc1bb5c05fa1803a8fbd7761770dc6d5fe', 'selection_seed_start': 21000000, 'test_seed_start': 22000000, 'base': {'format': 'triad-uno-linear-v1', 'features': ['play', 'number', 'skip', 'reverse', 'draw_two', 'wild', 'wild_draw_four', 'remaining_colour_count', 'remaining_colour_fraction', 'remaining_match_count', 'attack_next_threat', 'reverse_escape_threat', 'wild_when_small_hand', 'finish', 'keep', 'choose_colour', 'next_can_finish', 'own_hand_size'], 'weights': [8.022466349142684, 0.44153069585311683, -0.06834035646341297, 1.4643534535685507, -2.338619866848944, -3.4540620544700498, -4.276279368110987, 1.6602353386949849, 1.0165100023007814, 0.901341302040876, 3.853185081203389, 2.4436903583872347, 1.7223969671761579, 11.872419584625783, -4.598739912392293, 3.347870880522802, 0.32778952616834095, -0.24548152213113314], 'training': {'method': 'cross-entropy policy optimization', 'seed': 20261014, 'generations': 30, 'population': 32, 'games_per_candidate': 64, 'validation': 3.005859375, 'sdk_commit': '0cda3acc1bb5c05fa1803a8fbd7761770dc6d5fe', 'feature_count': 18}}}}
(work/"evaluation-incumbent.json").write_text(json.dumps(current_incumbent))
subprocess.run(["git","clone","--quiet","https://github.com/Sir-Nike/unoSDK.git",str(work/"unoSDK")],check=True)
subprocess.run(["git","checkout","--quiet","0cda3acc1bb5c05fa1803a8fbd7761770dc6d5fe"],cwd=work/"unoSDK",check=True)
os.environ["UNO_SDK"]=str(work/"unoSDK")
os.environ["OPENBLAS_NUM_THREADS"]="1"
os.environ["OMP_NUM_THREADS"]="1"
data=next(pathlib.Path("/kaggle/input").rglob("counterfactual.npz"))
command=["python",str(work/"train_uno_residual.py"),"--data",str(data),"--evaluation-incumbent",str(work/"evaluation-incumbent.json"),"--device","cuda","--width","64","--residual-scales","0.25,0.5,1","--workers","4","--validation-seed-start","50000000","--test-seed-start","51000000"]
subprocess.run(command+["--validation-seed-start","52000000","--test-seed-start","53000000","--epochs","3","--validation-blocks","4","--test-blocks","4","--output",str(work/"smoke-output")],check=True,timeout=600)
subprocess.run(command+["--epochs","100","--validation-blocks","256","--test-blocks","512","--output","/kaggle/working/output"],check=True,timeout=7200)
